In [1]:
import os, json, time, math, random, copy
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             precision_recall_fscore_support, confusion_matrix)
import matplotlib.pyplot as plt


SEED = 2026
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)


MODEL_NAME = "HooshvareLab/bert-fa-base-uncased"
MAX_LEN    = 128         
BATCH      = 64
EPOCHS     = 5
LR_ENCODER, LR_HEAD = 2e-5, 1e-3
WEIGHT_DECAY, WARMUP, CLIP = 0.01, 0.06, 1.0
EVAL_EVERY = 2000         
PATIENCE   = 3
LAMBDA_BAND = 1.0
FREEZE_ALL_BUT_LAST = False    


REC = ["recommended", "not_recommended", "no_idea"]
BAND = ["low", "medium", "high"]
BAND_CODE = {"low": 0, "medium": 1, "high": 2}
REC_ID = {c: i for i, c in enumerate(REC)}; BAND_ID = {c: i for i, c in enumerate(BAND)}


BASE_INPUT = Path("/kaggle/input")
DATA_DIR = BASE_INPUT / "train_sub" # Using this as the root just for print
OUT = Path("/kaggle/working"); OUT.mkdir(exist_ok=True)


dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Data root: {BASE_INPUT}")
print(f"Device: {dev}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: GPU not detected! Please enable it in Kaggle settings.")

Data root: /kaggle/input
Device: cuda
GPU: Tesla T4


In [9]:
def load(name):
    if "train_sub" in name:
        path = BASE_INPUT / "train_sub" / name
    elif "val_sub" in name:
        path = BASE_INPUT / "val_sub" / name
    elif "test" in name:
        path = BASE_INPUT / "test" / name
    else:
        path = BASE_INPUT / name
        
    return pd.read_csv(path, dtype=str, keep_default_na=False, na_values=[""])

train, val, test = load("train_sub.csv"), load("val_sub.csv"), load("test.csv")
print(f"Loaded rows -> Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")


tok = AutoTokenizer.from_pretrained(MODEL_NAME)


def encode(df):
    ids = []
    texts = df["body"].tolist()
    for i in range(0, len(texts), 20_000):
        ids += tok(texts[i:i+20_000], truncation=True, max_length=MAX_LEN,
                   add_special_tokens=True)["input_ids"]
    y_rec  = df["recommendation_status"].map(REC_ID).fillna(-100).astype(int).values
    y_band = df["rating_band"].map(BAND_ID).fillna(-100).astype(int).values
    return ids, y_rec, y_band


class DS(Dataset):
    def __init__(self, df): 
        self.ids, self.yr, self.yb = encode(df)
    def __len__(self): 
        return len(self.ids)
    def __getitem__(self, i): 
        return self.ids[i], self.yr[i], self.yb[i]


def collate(batch):
    L = max(len(b[0]) for b in batch)
    ids = torch.full((len(batch), L), tok.pad_token_id, dtype=torch.long)
    att = torch.zeros((len(batch), L), dtype=torch.long)
    for k, (x, _, _) in enumerate(batch):
        ids[k, :len(x)] = torch.tensor(x)
        att[k, :len(x)] = 1
    return ids, att, torch.tensor([b[1] for b in batch]), torch.tensor([b[2] for b in batch])

t0 = time.time()
ds_tr, ds_va, ds_te = DS(train), DS(val), DS(test)
print(f"Tokenized in {time.time()-t0:.0f}s")


lens = np.array([len(x) for x in ds_tr.ids[:50000]])
print(f"Truncated share (train sample): {(lens >= MAX_LEN).mean().round(3)}")


g = torch.Generator(); g.manual_seed(SEED)
dl_tr = DataLoader(ds_tr, batch_size=BATCH, shuffle=True, collate_fn=collate, generator=g,
                   num_workers=2, pin_memory=True)

def eval_loader(ds, bs=256):         
   
    order = np.argsort([len(x) for x in ds.ids])
    sub = torch.utils.data.Subset(ds, order)
    return DataLoader(sub, batch_size=bs, shuffle=False, collate_fn=collate, num_workers=2), order

dl_va, order_va = eval_loader(ds_va)
dl_te, order_te = eval_loader(ds_te)
print("DataLoaders are ready!")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/train_sub/train_sub.csv'

In [ ]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith(".csv"):
            print(os.path.join(root, f))